<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-solution/blob/main/notebooks/EcoMarket_AI_Solution.ipynb)


# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Taller Práctico #1
## Optimización de Atención al Cliente con IA Generativa

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

**Modelo:** `llama-3.3-70b-versatile` via [Groq API](https://console.groq.com)


In [1]:
import warnings
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages

In [2]:
!test "$IN_COLAB" = "True" && \
wget -q https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/requirements.txt -O requirements.txt && \
pip install -r requirements.txt

Cargar data

In [3]:
!test '{IN_COLAB}' = 'True' && mkdir -p data && [ ! -f data/orders_database.txt ] && wget -q -O data/orders_database.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/data/orders_database.txt

!test '{IN_COLAB}' = 'True' && mkdir -p data && [ ! -f data/return_policy.txt ] && wget -q -O data/return_policy.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/data/return_policy.txt

Cargar prompts

In [4]:
!test '{IN_COLAB}' = 'True' && mkdir -p prompts && [ ! -f prompts/system_prompts.txt ] && wget -q -O prompts/system_prompts.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/system_prompts.txt

!test '{IN_COLAB}' = 'True' && mkdir -p prompts && [ ! -f prompts/order_status_prompt.txt ] && wget -q -O prompts/order_status_prompt.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/order_status_prompt.txt

!test '{IN_COLAB}' = 'True' && mkdir -p prompts && [ ! -f prompts/return_policy_prompt.txt ] && wget -q -O prompts/return_policy_prompt.txt https://raw.githubusercontent.com/sebastianb92/ecomarket-solution/main/prompts/return_policy_prompt.txt

In [5]:
# ============================================================
# CONFIGURACIÓN SEGURA GROQ (COLAB + SECRETS)
# ============================================================

import os
from google.colab import userdata

def get_api_key():
    """
    Obtiene la API Key de forma segura desde Colab Secrets.
    Lanza error si no existe.
    """
    api_key = userdata.get("GROQ_API_KEY")

    if not api_key:
        raise ValueError("No se encontró GROQ_API_KEY en Colab Secrets")

    if not api_key.startswith("gsk_"):
        raise ValueError("La API Key no tiene formato válido")

    return api_key

# ========================
# VARIABLES DE CONFIG
# ========================

GROQ_API_KEY = get_api_key()

CONFIG = {
    "model": "llama-3.3-70b-versatile",
    "api_url": "https://api.groq.com/openai/v1/chat/completions",
    "max_tokens": 1024,
    "temperature": 0.3,
    "timeout": 30
}

# ========================
# HEADERS LISTOS
# ========================

HEADERS = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json"
}

print(f"API Key cargada correctamente ({GROQ_API_KEY[:6]}****)")
print(f"Modelo configurado: {CONFIG['model']}")

API Key cargada correctamente (gsk_Gw****)
Modelo configurado: llama-3.3-70b-versatile


In [6]:
# Carga de datos de ordenes y politicas

with open("data/orders_database.txt", "r", encoding="utf-8") as f:
    ORDERS_DATABASE = f.read()

with open("data/return_policy.txt", "r", encoding="utf-8") as f:
    RETURN_POLICY = f.read()

print("Datos cargados")

Datos cargados


In [7]:
# System Prompts

with open("prompts/system_prompts.txt", "r", encoding="utf-8") as f:
    ECOBOT_BASE = f.read()

with open("prompts/order_status_prompt.txt", "r", encoding="utf-8") as f:
    ORDER_SYSTEM = ECOBOT_BASE + "\n\n" + f.read()

with open("prompts/return_policy_prompt.txt", "r", encoding="utf-8") as f:
    RETURN_SYSTEM = ECOBOT_BASE + "\n\n" + f.read()

print("System prompts cargados")

System prompts cargados


In [9]:
# ============================================================
#  Cliente LLM
# ============================================================

import requests

def call_llm(system_prompt: str, user_message: str) -> str:
    """Llama a la API de Groq y retorna la respuesta generada."""

    if not GROQ_API_KEY:
        return "No se encontró la API Key. Revisa Colab Secrets."

    payload = {
        "model": CONFIG["model"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        "max_tokens": CONFIG["max_tokens"],
        "temperature": CONFIG["temperature"]
    }

    try:
        r = requests.post(
            CONFIG["api_url"],
            headers=HEADERS,
            json=payload,
            timeout=CONFIG["timeout"]
        )

        # ========================
        # MANEJO DE ERRORES
        # ========================
        if r.status_code == 401:
            return "API Key inválida. Verifica en console.groq.com"

        if r.status_code != 200:
            return f"Error HTTP {r.status_code}: {r.text[:200]}"

        data = r.json()

        # ========================
        # VALIDACIÓN RESPUESTA
        # ========================
        if "choices" not in data or len(data["choices"]) == 0:
            return "Respuesta inválida del modelo"

        return data["choices"][0]["message"]["content"].strip()

    except requests.exceptions.Timeout:
        return "Timeout: el modelo tardó demasiado en responder"

    except requests.exceptions.ConnectionError:
        return "Error de conexión: revisa tu internet"

    except Exception as e:
        return f"Error: {str(e)}"


print("Cliente LLM listo (adaptado a CONFIG)")

Cliente LLM listo (adaptado a CONFIG)


In [12]:
#  Funciones de Prompt Engineering

def query_order_status(tracking_number: str) -> str:
    """
    Consulta el estado de un pedido.
    Técnicas: RAG simulado, instrucciones paso a paso,
    manejo de casos extremos, restricción anti-alucinación.
    """
    user_prompt = f"""--- BASE DE DATOS DE PEDIDOS (usa ÚNICAMENTE esta información) ---
{ORDERS_DATABASE}
--- FIN DEL CONTEXTO ---

El cliente pregunta por su pedido: {tracking_number}

INSTRUCCIONES:
PASO 1 — Busca '{tracking_number}' en el contexto (mayúsculas/minúsculas, con y sin guión).

PASO 2.1 — Si el pedido EXISTE:
  - Estado actual con descripción clara
  - Fecha estimada de entrega o confirmación si ya fue entregado
  - URL de tracking si está disponible
  - Si RETRASADO: disculpa sincera primero, luego nueva fecha y razón
  - Si CANCELADO: estado del reembolso y plazos exactos
  - Si PENDIENTE DE PAGO: urgencia amable + enlace de pago
  - Próximo paso que el cliente debe esperar

PASO 2.1 — Si el pedido NO APARECE en el contexto:
  - Informa amablemente que no puedes encontrar ese número
  - Sugiere verificar el número (puede haber un error tipográfico)
  - Ofrece contacto: soporte@ecomarket.com o 900-ECO-MKT
  - NUNCA inventes datos de un pedido que no aparece en el contexto

FORMATO: Saludo personalizado + información + oferta de ayuda adicional."""
    return call_llm(ORDER_SYSTEM, user_prompt)


def request_return(product_name: str, reason: str, days: int,
                   opened: bool = False, first_return: bool = False) -> str:
    """
    Procesa una solicitud de devolución.
    Técnicas: árbol de decisión explícito, empatía antes de negativa,
    excepciones documentadas, nunca un 'no' sin alternativa.
    """
    opened_txt = "Sí" if opened else "No especificado"
    first_txt  = "Sí (posible excepción hasta 45 días)" if first_return else "No"

    user_prompt = f"""--- POLÍTICA DE DEVOLUCIONES DE ECOMARKET ---
{RETURN_POLICY}
--- FIN DE LA POLÍTICA ---

SOLICITUD DEL CLIENTE:
  Producto: {product_name}
  Motivo: {reason}
  Días desde la entrega: {days}
  ¿Producto abierto?: {opened_txt}
  ¿Primera devolución del cliente?: {first_txt}

ÁRBOL DE DECISIÓN (sigue en orden):

PASO 1 — Excepción universal:
  ¿El motivo menciona daño en envío, defecto o producto incorrecto?
  -> SÍ: APRUEBA siempre. Cubre el envío. Ofrece reembolso O reenvío. FIN.
  -> NO: continúa al Paso 2.

PASO 2 — Categoría del producto:
  ¿Es alimento, bebida, planta, suplemento? -> NO posible (sanitario). -> Paso 5.
  ¿Es higiene personal Y fue abierto? -> NO posible. -> Paso 5.
  ¿Es personalizado o digital? -> NO posible. -> Paso 5.
  Ninguna aplica -> puede devolverse. -> Paso 3.

PASO 3 — Plazo de 30 días:
  ¿Más de 30 días Y es primera devolución? -> Escala a agente humano, excepción posible.
  ¿Más de 30 días sin excepción? -> NO posible. -> Paso 5.
  ¿Dentro de 30 días? -> Devolución aprobada. -> Paso 4.

PASO 4 — RESPUESTA AFIRMATIVA:
  Confirma, explica el proceso en 3 pasos numerados, plazo del reembolso,
  ofrece crédito de tienda (+5% de bonificación).

PASO 5 — NEGATIVA EMPÁTICA:
  1. Empatía genuina primero (reconoce la frustración)
  2. Razón de forma simple y humana (no párrafo legal)
  3. Al menos UNA alternativa concreta
  4. NUNCA termines con un 'no' definitivo y frío sin ofrecer algo"""
    return call_llm(RETURN_SYSTEM, user_prompt)


def print_response(title: str, response: str):
    print(f"\n{'='*65}")
    print(f"📋 {title}")
    print('='*65)
    print(f"\n🤖 EcoBot:\n")
    print(response)
    print()

print("Funciones de prompt engineering listas")

Funciones de prompt engineering listas


---
## Ejercicio 1 — Consultas de Estado de Pedido
Demuestra el manejo de 5 estados distintos y la protección anti-alucinación.

In [13]:
# Pedido EN TRÁNSITO — información de entrega y tracking
response = query_order_status("ECO-12345")
print_response("ECO-12345 — Pedido EN TRÁNSITO — información de entrega y tracking", response)


📋 ECO-12345 — Pedido EN TRÁNSITO — información de entrega y tracking

🤖 EcoBot:

Hola María García,

Me alegra poder ayudarte con tu consulta sobre el estado de tu pedido ECO-12345. Después de verificar en nuestra base de datos, puedo informarte que tu pedido está actualmente EN TRÁNSITO. Esto significa que tu Kit de bambú reutilizable (3 piezas) ya ha sido despachado y se encuentra en camino hacia ti.

La fecha estimada de entrega para tu pedido es el 2024-07-05. Puedes seguir el progreso de tu pedido a través del enlace de seguimiento proporcionado por DHL Express: https://track.dhl.com/ECO12345. Nuestro último punto de control indica que tu pedido pasó por el Centro logístico Madrid.

Si tienes alguna otra pregunta o necesitas ayuda adicional, no dudes en hacérmelo saber. Estoy aquí para ayudarte. También puedes contactar con nuestro equipo de soporte a través del correo electrónico soporte@ecomarket.com o llamando al 900-ECO-MKT (L-V 9:00-18:00h) si prefieres.

¿Hay algo más en lo

In [14]:
# Pedido RETRASADO — debe incluir disculpa empática primero
response = query_order_status("ECO-12346")
print_response("ECO-12346 — Pedido RETRASADO — debe incluir disculpa empática primero", response)


📋 ECO-12346 — Pedido RETRASADO — debe incluir disculpa empática primero

🤖 EcoBot:

Hola Carlos,

Me dirijo a ti porque he encontrado un pedido asociado a tu nombre con el número ECO-12346. Desafortunadamente, tu pedido está retrasado. Quiero empezar expresando mi más sincera disculpa por las molestias que esto te pueda haber causado. Entendemos que esperar un poco más de lo esperado puede ser frustrante.

La nueva fecha estimada de entrega para tu pedido, que incluye un jabón orgánico artesanal (pack x6), es el 2024-07-08. El retraso se debe a una huelga de transporte regional en Cataluña, que ha afectado a varios de nuestros envíos. Puedes seguir el progreso de tu pedido a través del enlace de seguimiento: https://track.correos.es/ECO12346.

Queremos asegurarte que estamos haciendo todo lo posible para minimizar el impacto de este retraso y para que puedas recibir tu pedido lo antes posible. Si tienes alguna pregunta o necesitas más información, no dudes en hacérmelo saber.

¿Hay al

In [27]:
# Pedido ENTREGADO — Confirmación de entrega
response = query_order_status("ECO-12347")
print_response("ECO-12347 — Pedido ENTREGADO — Confirmación de entrega", response)


📋 ECO-12347 — Pedido ENTREGADO — Confirmación de entrega

🤖 EcoBot:

Hola Ana Martínez,

Me alegra poder ayudarte con tu consulta sobre el estado de tu pedido ECO-12347. Según nuestra base de datos, tu pedido ya ha sido entregado con éxito. La entrega se realizó el 2024-06-25 a las 14:32h, y se ha registrado una firma digital como confirmación de recepción.

Puedes verificar el estado de tu pedido en el enlace de seguimiento: https://track.seur.com/ECO12347. Si tienes alguna pregunta o inquietud sobre tu pedido, no dudes en hacérmelo saber.

En EcoMarket, nos comprometemos con la sostenibilidad y la satisfacción de nuestros clientes. Si necesitas ayuda adicional o tienes alguna otra consulta, estoy aquí para ayudarte. ¿Hay algo más en lo que pueda asistirte? Puedes responder a este mensaje o contactarnos a través de soporte@ecomarket.com o llamando al 900-ECO-MKT. Estamos aquí para ayudarte.



In [15]:
# Pedido CANCELADO — debe informar el estado del reembolso
response = query_order_status("ECO-12349")
print_response("ECO-12349 — Pedido CANCELADO — debe informar el estado del reembolso", response)


📋 ECO-12349 — Pedido CANCELADO — debe informar el estado del reembolso

🤖 EcoBot:

Hola Sofia,

Me dirijo a ti porque he encontrado que el pedido ECO-12349 está asociado a tu nombre en nuestra base de datos. Quiero informarte que tu pedido ECO-12349 ha sido cancelado según tu solicitud. La cancelación se procesó el 2024-07-01 y el reembolso ya ha sido iniciado. Debes recibir el reembolso en un plazo de 5-7 días hábiles a la tarjeta original terminada en ****4521.

Quiero asegurarme de que estás informada de todos los detalles relevantes. Si tienes alguna pregunta o necesitas más información sobre el estado de tu reembolso, no dudes en hacérmelo saber.

¿Hay algo más en lo que pueda ayudarte? Estoy aquí para asistirte con cualquier consulta o inquietud que puedas tener. Si necesitas ayuda adicional, también puedes contactar nuestro equipo de soporte en soporte@ecomarket.com o llamando al 900-ECO-MKT. Estamos comprometidos con la sostenibilidad y con brindarte la mejor experiencia posib

In [16]:
# Pedido PENDIENTE DE PAGO — urgencia amable
response = query_order_status("ECO-12351")
print_response("ECO-12351 — Pedido PENDIENTE DE PAGO — urgencia amable", response)


📋 ECO-12351 — Pedido PENDIENTE DE PAGO — urgencia amable

🤖 EcoBot:

Hola Isabella,

Me alegra ayudarte con tu consulta sobre el pedido ECO-12351. Después de verificar en nuestra base de datos, encontré que tu pedido está en estado "PENDIENTE DE PAGO". Esto significa que el pago no fue confirmado y es importante que completes el pago lo antes posible para evitar la cancelación del pedido.

La fecha de pedido fue el 2024-07-03, y es crucial que completes el pago en las próximas 24 horas para que podamos procesar tu pedido sin retrasos. Puedes completar el pago a través del siguiente enlace: https://ecomarket.com/pago/ECO12351.

Es importante que completes el pago lo antes posible para que podamos enviar tu pedido de Velas aromáticas de cera de soja (x3) sin demoras. Si tienes alguna pregunta o necesitas ayuda con el proceso de pago, no dudes en hacérmelo saber.

¿Hay algo más en lo que pueda ayudarte? Estoy aquí para asegurarme de que tengas una experiencia de compra satisfactoria en E

In [17]:
# Número INEXISTENTE — prueba anti-alucinación
response = query_order_status("ECO-99999")
print_response("ECO-99999 — Número INEXISTENTE — prueba anti-alucinación", response)


📋 ECO-99999 — Número INEXISTENTE — prueba anti-alucinación

🤖 EcoBot:

Hola, lamentablemente no puedo encontrar el pedido ECO-99999 en nuestra base de datos. Es posible que haya un error tipográfico en el número de pedido. Te sugiero verificar el número de pedido nuevamente para asegurarte de que esté correcto.

Si después de verificar el número de pedido, sigues teniendo problemas para encontrar información sobre tu pedido, no dudes en contactarnos. Puedes escribirnos a soporte@ecomarket.com o llamar al 900-ECO-MKT (gratuito, L-V 9:00-18:00h) para obtener ayuda adicional. Estamos aquí para ayudarte con cualquier pregunta o inquietud que tengas sobre tus pedidos en EcoMarket.

¿Hay algo más en lo que pueda ayudarte mientras tanto? Estoy aquí para ayudar con cualquier otra consulta que tengas.



---
## Ejercicio 2 — Solicitudes de Devolución
El desafío: responder con empatía tanto el «sí» como el «no»,
y nunca dar una negativa sin ofrecer una alternativa.

In [18]:
# Devolución APROBADA — producto de hogar dentro del plazo
response = request_return(product_name="Botella de acero inoxidable 750ml", reason="No me gusta el diseño", days=12, opened=False)
print_response("Devolución APROBADA — producto de hogar dentro del plazo", response)


📋 Devolución APROBADA — producto de hogar dentro del plazo

🤖 EcoBot:

Hola, gracias por contactarnos en EcoMarket. Entiendo que deseas devolver la botella de acero inoxidable 750ml porque no te gusta el diseño. Me parece completamente razonable que quieras un producto que se adapte a tus gustos y necesidades.

Después de revisar nuestra política de devoluciones, puedo confirmar que la botella de acero inoxidable se encuentra dentro de las categorías que pueden ser devueltas. Dado que has recibido el producto hace 12 días y no has mencionado que esté dañado o tenga defectos, podemos proceder con la devolución.

Aquí te explico el proceso de devolución en 3 pasos sencillos:

1. **Solicita la devolución**: Ya has dado el primer paso al contactarnos. Ahora, te pediré que accedas a nuestra página de devoluciones en ecomarket.com/devoluciones, donde deberás introducir tu número de pedido y motivo de devolución.
2. **Recibe la etiqueta de envío**: En un plazo de 24 horas hábiles, recibirás 

In [21]:
# Devolución RECHAZADA — higiene personal abierto
response = request_return(product_name="Jabón orgánico artesanal", reason="No me gusta el olor", days=5, opened=True)
print_response("Devolución RECHAZADA — higiene personal abierto", response)


📋 Devolución RECHAZADA — higiene personal abierto

🤖 EcoBot:

Entiendo perfectamente tu situación, y me imagino que debe ser frustrante no gustarte el olor del jabón orgánico artesanal después de abrirlo. Me gustaría ayudarte a encontrar una solución.

Desafortunadamente, según nuestra política de devoluciones, los productos de higiene personal que han sido abiertos no pueden ser devueltos, ya que no podemos garantizar la higiene y seguridad de estos productos una vez que han sido utilizados o abiertos.

Sin embargo, no quiero dejar la situación así. Me gustaría ofrecerte una alternativa. Aunque no podemos aceptar la devolución del jabón, podríamos considerar ofrecerte un descuento o un cupón de compensación para tu próxima compra en EcoMarket. Esto podría ser una forma de compensarte por la inconveniencia y ayudarte a encontrar un producto que mejor se adapte a tus necesidades.

Si estás interesado en explorar esta opción, puedo conectarte con uno de nuestros agentes de atención al c

In [22]:
# Devolución RECHAZADA — producto perecedero (alimento)
response = request_return(product_name="Mix de frutos secos orgánicos", reason="Cambié de opinión", days=3, opened=False)
print_response("Devolución RECHAZADA — producto perecedero (alimento)", response)


📋 Devolución RECHAZADA — producto perecedero (alimento)

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver un producto, específicamente un mix de frutos secos orgánicos, debido a un cambio de opinión. Me gustaría ayudarte con este proceso.

Desafortunadamente, según nuestra política de devoluciones, los alimentos y productos perecederos no pueden ser devueltos una vez enviados, por razones de seguridad alimentaria e higiene. Esto incluye los frutos secos orgánicos que compraste.

Entiendo que esto puede ser frustrante, especialmente si has cambiado de opinión sobre el producto. Quiero asegurarte que estamos aquí para ayudarte y encontrar una solución que se adapte a tus necesidades.

Aunque no podemos aceptar la devolución de este producto, te ofrecemos una alternativa. Si deseas, podemos proporcionarte un cupón de descuento del 10% para tu próxima compra en EcoMarket. Esto te permitirá elegir otro producto que se adapte mejor a tus necesidades y prefe

In [23]:
# Devolución APROBADA — daño en tránsito (excepción universal)
response = request_return(product_name="Cepillo de dientes de bambú", reason="El paquete llegó aplastado y roto", days=4, opened=True)
print_response("Devolución APROBADA — daño en tránsito (excepción universal)", response)


📋 Devolución APROBADA — daño en tránsito (excepción universal)

🤖 EcoBot:

Hola, gracias por contactarnos sobre el cepillo de dientes de bambú que llegó dañado. Entiendo perfectamente tu frustración al recibir un producto en mal estado, especialmente cuando se trata de algo que esperabas recibir en perfectas condiciones.

En este caso, dado que el producto llegó aplastado y roto, podemos considerar una excepción a nuestra política de devoluciones. Aunque el producto es de higiene personal y fue abierto, el daño en el envío es una situación que queremos resolver de manera justa.

**APROBACIÓN DE DEVOLUCIÓN**: Dado el daño en el envío, podemos aprobar la devolución de tu cepillo de dientes de bambú. Aquí te explico el proceso:

1. **Solicita la devolución**: Ya has dado el primer paso al contactarnos. Ahora, te pedimos que accedas a nuestra página de devoluciones en ecomarket.com/devoluciones, introduces tu número de pedido y motivo de devolución.
2. **Recibe la etiqueta de envío**: En 

In [24]:
# Devolución FUERA DE PLAZO — primera vez, escala a agente
response = request_return(product_name="Set de cubiertos de bambú", reason="Me lo regalaron y no lo necesito", days=38, opened=False, first_return=True)
print_response("Devolución FUERA DE PLAZO — primera vez, escala a agente", response)


📋 Devolución FUERA DE PLAZO — primera vez, escala a agente

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que has recibido un set de cubiertos de bambú como regalo y no lo necesitas. Me parece una situación un poco incómoda, ya que no es algo que hayas elegido tú mismo.

Sin embargo, veo que han pasado 38 días desde la entrega, lo que supera el plazo de 30 días establecido para las devoluciones. Como es tu primera devolución, hay una excepción posible que podría aplicarse. Me gustaría escalar tu caso a uno de nuestros agentes humanos para que puedan evaluar si se puede hacer una excepción en tu caso.

Antes de hacerlo, quiero asegurarme de que entiendas que, si se aprueba la devolución, el proceso sería el siguiente:

1. Debes solicitar la devolución en nuestra página web, introduciendo tu número de pedido y motivo de devolución.
2. Recibirás una etiqueta de envío prepagada por email, que te permitirá enviar el producto de vuelta a nosotros.
3. Una vez que recibamos e

---
## Modo Interactivo
Cambia los valores de las variables y ejecuta las celdas para probar con tus propios datos.

In [25]:
# CONSULTA DE PEDIDO PERSONALIZADA
# Pedidos disponibles: ECO-12345 al ECO-12354, o prueba uno inexistente
MI_PEDIDO = "ECO-12353"  # <- cambia aquí

response = query_order_status(MI_PEDIDO)
print_response(f"Consulta personalizada: {MI_PEDIDO}", response)


📋 Consulta personalizada: ECO-12353

🤖 EcoBot:

Hola Valentina,

Me alegra ayudarte con tu consulta sobre el pedido ECO-12353. Después de verificar en nuestra base de datos, encontré que tu pedido está en el estado "LISTO PARA RECOGIDA EN TIENDA". Esto significa que tu producto, el contenedor de vidrio hermético (set x5), ya está disponible para que lo recojas en nuestra tienda ubicada en C/ Gran Vía 45, Local 2, en Madrid.

La recogida está disponible hasta el 2024-07-10, y el horario de recogida es de lunes a sábado de 10:00 a 20:00h. Por favor, no olvides presentar tu DNI o el número de pedido en la tienda para poder recoger tu compra de manera efectiva.

Si tienes alguna pregunta adicional o necesitas ayuda con la recogida, no dudes en hacérmelo saber. Estoy aquí para ayudarte. Además, si necesitas cualquier otra asistencia o tienes alguna otra consulta, puedes contactarnos a través de nuestro email soporte@ecomarket.com o llamando al 900-ECO-MKT.

¿Hay algo más en lo que pueda ay

In [26]:
# SOLICITUD DE DEVOLUCIÓN PERSONALIZADA
MI_PRODUCTO  = "Velas aromáticas de cera de soja"  # <- cambia aquí
MI_MOTIVO    = "No me gusta el aroma"               # <- cambia aquí
DIAS         = 8                                    # <- días desde la entrega
FUE_ABIERTO  = False                                # <- True o False
PRIMERA_VEZ  = False                                # <- True si es primera devolución

response = request_return(MI_PRODUCTO, MI_MOTIVO, DIAS, FUE_ABIERTO, PRIMERA_VEZ)
print_response(f"Devolución personalizada: {MI_PRODUCTO}", response)


📋 Devolución personalizada: Velas aromáticas de cera de soja

🤖 EcoBot:

Hola, gracias por contactar con EcoMarket. Entiendo que deseas devolver las velas aromáticas de cera de soja porque no te gusta el aroma. Me parece una razón perfectamente válida para querer devolver un producto.

Después de revisar nuestra política de devoluciones, puedo confirmarte que las velas aromáticas de cera de soja sí pueden devolverse, ya que están dentro de la categoría de productos de hogar y cocina, y no has mencionado que hayan sido abiertas o utilizadas.

Aquí te explico el proceso de devolución en 3 pasos sencillos:

1. **Solicita la devolución**: Debes acceder a nuestra página web en [ecomarket.com/devoluciones](http://ecomarket.com/devoluciones), introducir tu número de pedido y el motivo de la devolución. Ten en cuenta que debes hacerlo dentro de los 30 días naturales desde la entrega.
2. **Recibe la etiqueta de envío**: En un plazo de 24 horas hábiles, recibirás por correo electrónico una etiq